# Lección 03 - Patrones de Diseño Agéntico

En esta lección, exploramos tres patrones de diseño fundamentales para construir agentes de IA efectivos:

1. **Instrucciones Claras para el Agente** — Crear indicaciones precisas que definan el rol y guíen el comportamiento del agente
2. **Salida Estructurada con Modelos Pydantic** — Asegurar que los agentes devuelvan datos predecibles y validados
3. **Agentes de Responsabilidad Única** — Diseñar agentes enfocados que hagan bien una sola cosa cada uno

Aplicaremos cada patrón a un escenario de **recomendador de destinos de viaje**, construyendo progresivamente un sistema que pueda sugerir destinos, verificar disponibilidad y manejar la logística.


## Configuración


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Patrón 1: Instrucciones Claras para el Agente

El patrón más impactante también es el más simple: escribir instrucciones claras y detalladas para tu agente.

Buenas instrucciones definen:
- **Quién** es el agente (personalidad y tono)
- **Qué** debe hacer (responsabilidades paso a paso)
- **Cómo** debe comportarse (restricciones y estilo)

A continuación, creamos un agente conserje de viajes con instrucciones explícitas que moldean cada respuesta que produce.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Patrón 2: Salida estructurada con modelos Pydantic

El texto libre es útil para la conversación, pero los sistemas posteriores necesitan datos estructurados.
Al combinar **modelos Pydantic** con una **función herramienta**, podemos:

- Definir un esquema exacto para la salida del agente
- Validar respuestas automáticamente
- Integrar los resultados del agente en la lógica de la aplicación de forma fiable

La clave para la aplicación es pasar `response_format` cuando ejecutamos el agente. Esto obliga al
modelo a devolver un objeto `TravelRecommendations` validado (disponible en `response.value`)
en lugar de texto libre. La herramienta `get_destination_details` también devuelve un
`DestinationRecommendation` tipado, por lo que los datos permanecen estructurados de principio a fin.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Patrón 3: Agentes de Responsabilidad Única

Las tareas complejas se benefician de dividir el trabajo entre múltiples agentes enfocados, cada uno con una única responsabilidad:

- Un **Experto en Destinos** que conoce sobre lugares y disponibilidad
- Un **Planificador de Logística** que maneja vuelos, hoteles e itinerarios

Esto refleja el principio de ingeniería de software de *separación de responsabilidades* — cada agente es más fácil de probar, mantener y mejorar de forma independiente.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Resumen

En esta lección aplicamos tres patrones de diseño agente a un escenario de recomendación de viajes:

| Patrón | Idea Clave | Beneficio |
|---|---|---|
| **Instrucciones Claras** | Definir persona, responsabilidades y restricciones desde el principio | Comportamiento del agente consistente y alineado con la marca |
| **Salida Estructurada** | Usar modelos Pydantic como formato de respuesta | Resultados validados y legibles por máquina |
| **Responsabilidad Única** | Dar a cada agente un trabajo enfocado | Más fácil de probar, mantener y componer |

Estos patrones se combinan de forma natural: puedes unir instrucciones claras con salida estructurada dentro de un agente de responsabilidad única para construir sistemas robustos y listos para producción.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
